In [5]:
# external imports
import ast
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from pathlib import Path
from typing import List
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

def load_all_csvs(csv_paths: List[str]) -> pd.DataFrame:
    """Load and combine multiple experiment CSVs. Do simple data processing for later convenience"""
    dfs = []
    for path in csv_paths:
        df = pd.read_csv(path, index_col=0)
        df['source_file'] = Path(path).parent.name  # track which experiment it came from
        dfs.append(df)
    
    combined = pd.concat(dfs, ignore_index=True)
    
    # add in diff representations of red params for convenience
    combined[['resizeTrigger', 'sizeLimit']] = combined['reduce_triggerSz_sizeLim'].str.strip('()').str.split(',', expand=True).astype(int)
    combined['gap_size_range_tuple'] = combined['gap_size_range'].apply(ast.literal_eval)
    combined['reduce_triggerSz_sizeLim_tuple'] = combined['reduce_triggerSz_sizeLim'].apply(ast.literal_eval)
    combined['time_norm'] = combined['sumMetrics_time_mean'] / combined['sumMetrics_time_mean'].max()
    combined['coverage_norm'] = combined['result_coverage_mean'] / combined['result_coverage_mean'].max()
    combined['efficiency'] = combined['coverage_norm'] / combined['time_norm']
    return combined

In [10]:
csv = ['data/results/2358055190/ni_sweeping100/ni15_red500_250_sweep/results_sd2358055190.csv', 'data/results/2358055190/ni_sweeping100/ni15_red500_100_sweep/results_sd2358055190.csv', 'data/results/2358055190/ni_sweeping100/ni15_red500_10_sweep/results_sd2358055190.csv', 'data/results/2358055190/ni_sweeping100/ni15_red150_100_sweep/results_sd2358055190.csv', 'data/results/2358055190/ni_sweeping100/ni15_red150_10_sweep/results_sd2358055190.csv', 'data/results/2358055190/ni_sweeping100/ni15_red70_50_sweep/results_sd2358055190.csv', 'data/results/2358055190/ni_sweeping100/ni15_red15_10_sweep/results_sd2358055190.csv', 'data/results/2358055190/ni_sweeping100/ni15_red3_1_sweep/results_sd2358055190.csv']

df = load_all_csvs(csv)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 52 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   master_seed                      56 non-null     int64  
 1   data_type                        56 non-null     object 
 2   num_trials                       56 non-null     int64  
 3   dataset_size                     56 non-null     int64  
 4   uncertain_ratio                  56 non-null     float64
 5   interval_size_range              56 non-null     object 
 6   mult_size_range                  56 non-null     object 
 7   num_intervals                    56 non-null     int64  
 8   start_interval_range             56 non-null     object 
 9   gap_size                         0 non-null      float64
 10  interval_width                   0 non-null      float64
 11  num_intervals_range              0 non-null      float64
 12  gap_size_range          

In [19]:
class a:
    def __init__(self):
        self.iv = "num_intervals"
        self.resultFilepath = "./"
    
    def plot_efficiency_vs_iv(self, df: pd.DataFrame, filename="efficiency_vs_iv.pdf"):
        iv = self.iv 
        filename = f"{self.resultFilepath}/{filename}"
        
        # if iv == "gap_size" or iv == "gap_size_range":
        if iv == "gap_size_range":
            iv = 'gap_size_range_tuple'

        with PdfPages(filename) as pdf:
            # heatmap: reduce_triggers(x) vs IV(y)
            pivot_eff = df.pivot_table(
                index=iv,
                columns='reduce_triggerSz_sizeLim',
                values='efficiency',
                aggfunc='mean'
            )
        
            
            plt.figure(figsize=(6,3))
            sns.heatmap(pivot_eff, annot=True, fmt=".2f", cmap="coolwarm", annot_kws={"size": 6})
            plt.title("Efficiency (coverage_norm/time_norm)")
            plt.xlabel("reduce_triggerSz_sizeLim")
            plt.ylabel(iv)
            pdf.savefig(bbox_inches='tight')
            plt.close()
            
            iv_values = sorted(df[iv].unique())
            red_configs = sorted(df['reduce_triggerSz_sizeLim'].unique())
            fig, axes = plt.subplots(1, len(iv_values), figsize=(4*len(iv_values), 3))
            axes = axes.flatten() if len(iv_values) > 1 else [axes]
            # fig.text(0.5, 0.05, self.param_str, ha='center', va='center', fontsize=7, color='black', wrap=True)

            handles, labels = None, None
            for i, val in enumerate(iv_values):
                ax = axes[i]
                sub_df = df[df[iv] == val]
                for r in red_configs:
                    r_df = sub_df[sub_df['reduce_triggerSz_sizeLim'] == r]
                    ax.scatter(r_df['time_norm'], r_df['coverage_norm'], label=str(r), s=50)

                if handles is None:
                    handles, labels = ax.get_legend_handles_labels()

                ax.set_title(f'{iv}={val}')
                ax.set_xlabel('relative time')
                ax.set_ylabel('relative coverage')
                ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=False, useOffset=False))
                ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=False, useOffset=False))
                ax.grid(True, alpha=0.3)

            plt.tight_layout(rect=[0, 0.08, 0.85, 1])
            pdf.savefig(bbox_inches='tight')
            plt.close()

            fig = plt.figure(figsize=(6,2))
            fig.legend(handles, labels, title='(trigger, size_limit)', loc='center', fontsize=8)
            pdf.savefig(bbox_inches='tight')
            plt.close()

In [20]:
t = a()
t.plot_efficiency_vs_iv(df)

ValueError: zero-size array to reduction operation fmin which has no identity

<Figure size 600x300 with 0 Axes>

KeyError: "None of [Index([ 1,  2,  3,  4,  6, 10, 15,  1,  2,  3,  4,  6, 10, 15,  1,  2,  3,  4,\n        6, 10, 15,  1,  2,  3,  4,  6, 10, 15,  1,  2,  3,  4,  6, 10, 15,  1,\n        2,  3,  4,  6, 10, 15,  1,  2,  3,  4,  6, 10, 15,  1,  2,  3,  4,  6,\n       10, 15],\n      dtype='int64')] are in the [columns]"